In [30]:
import numpy as np
import pandas as pd
%matplotlib widget
import matplotlib.pyplot as plt
import os
import sys
import h5py
from biosppy.signals import ecg
sys.path.append('/home/wolfgang/repos/sleep_research_io')
from sleep_research_functions import load_sleep_data, write_to_hdf5_file, get_metadata, read_in_routine
import pickle
import matlab.engine
import matlab
import scipy.signal
import datetime
import matplotlib.dates as mdates
from HRV_and_CPC_analysis_functions import ecg_hrv_cpc_routine, spectrum_plot_dt, cpc_and_signals_plot

In [39]:
ecg_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG'
vitals_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_v2'

# save folders:
cpc_spectrograms_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/CPC_spectrograms'
ecg_r_peak_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/BMR_resampled_ECG_and_rpeaks'
vitals_hrv_dir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_v2_hrv'


In [5]:
vitals_but_no_ecg = list(set(vitals_files_available) - set(ecg_files_available))
vitals_but_no_ecg.sort()
print(f'10Hz vitals sign dataframe available but no ECG: {len(vitals_but_no_ecg)} files.')
print(vitals_but_no_ecg)
both_available = list(set(vitals_files_available).intersection(set(ecg_files_available)))
print(f'\nBoth vitals and ECG available: {len(both_available)}')
study_ids = both_available
study_ids.sort()

10Hz vitals sign dataframe available but no ECG: 78 files.
['014', '018', '031', '033', '034', '035', '037', '039', '042', '045', '046', '047', '048', '049', '050', '051', '052', '054', '056', '061', '092', '093', '095', '097', '100', '101', '104', '105', '106', '107', '108', '109', '110', '111', '112', '114', '115', '117', '119', '120', '121', '122', '123', '124', '125', '142', '143', '149', '151', '152', '153', '154', '155', '156', '157', '158', '160', '161', '164', '165', '167', '169', '171', '172', '173', '175', '176', '177', '178', '179', '180', '181', '182', '183', '185', '186', '188', '189']

Both vitals and ECG available: 34


In [51]:
study_id = 126

# savepaths for this study_id:
cpc_spectrogram_path = os.path.join(cpc_spectrograms_dir, 'CPC_spectrogram_' + str(study_id).zfill(3) + '.csv')
ecg_rpeak_path = os.path.join(ecg_r_peak_dir, 'ECG_' + str(study_id).zfill(3) + '.h5')
vitals_hrv_path = os.path.join(vitals_hrv_dir, f'icusleep_{str(study_id).zfill(3)}.h5')


In [7]:
path_ecg_data = os.path.join(ecg_dir, f'ECG_{study_id}.h5')
path_vitals_data = os.path.join(vitals_dir, f'icusleep_{study_id}.h5')

signals_contained_ecg = get_metadata(path_ecg_data)
signals_contained_vitals = get_metadata(path_vitals_data)

In [8]:
fs_hrv = 10
hrv_10hz, signal_ecg_lead, cpc_df = ecg_hrv_cpc_routine(path_ecg_data, path_vitals_data, fs_hrv=fs_hrv)

Attention, only subset of data selected. DEBUG/ DEVELOP MODE.


/home/wolfgang/repos/ICU-Sleep/code1/HRV_and_CPC_analysis_functions.py:317: RuntimeWarning: invalid value encountered in less
  C[C<0.3] = 0


In [9]:
# save CPC spectrograms in own folder:
cpc_df.to_csv(cpc_spectrogram_path) # index: DateTime, columnnames: frequency.

In [ ]:
# save ECG and r peak detection:

hdr = {}
dt = signal_ecg_lead.index[0]
hdr['start_datetime'] = np.array([dt.year, dt.month, dt.day,
         dt.hour, dt.minute,
         dt.second, dt.microsecond])
hdr['study_id'] = int(study_id)
hdr['fs'] = 240

write_to_hdf5_file(signal_ecg_lead, ecg_rpeak_path, hdr = hdr, default_dtype='float16', overwrite=False)

In [33]:
vitals_data, hdr, fs = read_in_routine(path_vitals_data)

hrv_10hz.rename({'signal' : 'ecg_signal'}, axis=1, inplace=True)
vitals_data = vitals_data.join(hrv_10hz, how='outer')

write_to_hdf5_file(vitals_data, vitals_hrv_path, hdr = hdr, default_dtype='float16', overwrite=False)


In [50]:
# signal_ecg_lead, hdr = load_sleep_data(ecg_rpeak_path, idx_to_datetime=1)

# vitals_hrv, hdr, fs = read_in_routine(vitals_hrv_path)

In [9]:
# plt.figure(read_in_routinet(signal_ecg_lead.signal)
# plt.scatter(signal_ecg_lead.index[signal_ecg_lead.r_peak==1], signal_ecg_lead.signal[signal_ecg_lead.r_peak==1], c='r', s=8)
# plt.show()

In [10]:
# print('CPC')
# print(np.nanmean(cpc))
# print(np.nanstd(cpc))
# print(np.nanmax(cpc))
# print('Spectrum Resp')
# print(np.nanmean(spec_resp))
# print(np.nanstd(spec_resp))
# print(np.nanmax(spec_resp))
# print('Spectrum NN')
# print(np.nanmean(spec_nn))
# print(np.nanstd(spec_nn))
# print(np.nanmax(spec_nn))
# print('Crossspectrum')
# print(np.nanmean(crossspectrum))
# print(np.nanstd(crossspectrum))
# print(np.nanmax(crossspectrum))
# print('Coherence')
# print(np.nanmean(coherence))
# print(np.nanstd(coherence))
# print(np.nanmax(coherence))


In [ ]:
### PLOT:

signals_to_load = ['movavg_0_5s']
resp, hdr = load_sleep_data(path_vitals_data, load_all_signals=0, signals_to_load=signals_to_load, idx_to_datetime=1)
hfc_lfc_ratio = hrv_10hz[['hfc_lfc_ratio']]
cpc_and_signals_plot(cpc_df, hfc_lfc_ratio, resp, hrv_10hz['nn'])


In [111]:
hrv_10hz.loc['2019-07-30 09:49:15':];

In [115]:
hfc_lfc = hrv_10hz.loc[hrv_10hz.cpc_cntr_window==1, ['hfc_lfc_ratio']]

In [111]:
for i in range(100):
    plt.close()

In [104]:
fig, ax = plt.subplots(1,1, figsize=(10,2))
ax.plot(resp_nn.resp)
ax.plot(100*(cpc_df.lfc))
ax.plot(1000*(cpc_df.hfc))
ax.plot(cpc_df.hfc_lfc_ratio)
plt.legend(['Respiration', 'LFC', 'HFC', 'HFC_LFC Ratio'], bbox_to_anchor=[1.0, 0.5])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [137]:
fig, ax = plt.subplots(1,1, figsize=(10,2))
ax.plot(resp_nn.resp)
ax.plot((cpc_df.lfc/10))
ax.plot((cpc_df.hfc))
ax.plot(cpc_df.hfc_lfc_ratio)
plt.legend(['Respiration', 'LFC', 'HFC', 'HFC_LFC Ratio'], bbox_to_anchor=[1.0, 0.5])

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [ ]:
def ecg_peak_detection_plot():

    sel = signal_ecg_lead.iloc[start_idx : start_idx + duration*fs]
    plt.figure(figsize=(13,4))
    plt.plot(sel.signal)
#     plt.plot(sel.filtered)
    plt.scatter(sel.loc[sel['r_peak']==1].index, sel.signal[sel['r_peak']==1], zorder=10, c='r')
#     plt.scatter(sel.loc[sel['r_peak_adarri']==1].index, sel.filtered[sel['r_peak_adarri']==1], zorder=10, c='g')
    plt.tight_layout()

if 0:
    start_idx = 10000
    duration = 40
    ecg_peak_detection_plot()

    start_idx = 100000
    duration = 40
    ecg_peak_detection_plot()

    start_idx = 500000
    duration = 40
    ecg_peak_detection_plot()

    start_idx = 1000000
    duration = 40
    ecg_peak_detection_plot()

    start_idx = 3000000
    duration = 40
    ecg_peak_detection_plot()


In [19]:
# USING BIOSPPY ECG MODULE:

if 0:

    if 1:
        import time
        a = time.time()
        out_ecg = ecg.ecg(signal=signal, sampling_rate=fs, show=False)
        with open(f'out_ecg_{study_id}.p', 'wb') as f:
            pickle.dump(out_ecg, f, protocol=4)

        print(time.time() - a)
    else:
        todo = 1 


    # print(out_ecg['ts'].shape)
    # print(out_ecg['filtered'].shape)
    # print(out_ecg['rpeaks'].shape)
    # print(out_ecg['templates_ts'].shape)
    # print(out_ecg['templates'].shape)
    # print(out_ecg['heart_rate_ts'].shape)
    # print(out_ecg['heart_rate'].shape)


    signal_ecg_lead['filtered'] = out_ecg['filtered']
    signal_ecg_lead['r_peak'] = 0
    signal_ecg_lead.loc[signal_ecg_lead.iloc[out_ecg['rpeaks']].index, 'r_peak'] = 1

    rpeaks_noart = adarri_artifact_removal(
        out_ecg["rpeaks"], signal
    )
    rr_interval = np.diff(rpeaks_noart) / fs  # In seconds

    signal_ecg_lead['r_peak_adarri'] = 0
    signal_ecg_lead.loc[signal_ecg_lead.iloc[rpeaks_noart].index, 'r_peak_adarri'] = 1



46.222904205322266


In [40]:
#### investigate difference in ECG peak detection:

if 0:

    removed_by_adarri = list(set(out_ecg["rpeaks"]) - set(rpeaks_noart))
    removed_by_adarri.sort()
    removed_by_adarri = np.array(removed_by_adarri)
    removed_by_adarri = removed_by_adarri[np.where(np.diff(removed_by_adarri)>240)[0]]

    physio = np.round(nn[:,0]*fs).astype(int)

    physio_alone = []
    for cand in physio:
        cand_env = np.arange(cand-20,cand+20)
        if all(np.logical_not(np.isin(cand_env, rpeaks_noart))):
            physio_alone.append(cand)

    physio_missing = []
    for cand in rpeaks_noart:
        cand_env = np.arange(cand-20,cand+20)
        if all(np.logical_not(np.isin(cand_env, physio))):
            physio_missing.append(cand)

    def ecg_peak_detection_plot_compare1():

        sel = signal_ecg_lead.iloc[start_idx : start_idx + duration*fs]
        plt.figure(figsize=(13,4))
        plt.plot(sel.signal)
        plt.plot(sel.filtered)
        plt.scatter(sel.loc[sel['r_peak']==1].index, sel.filtered[sel['r_peak']==1], zorder=10, c='r')
        plt.scatter(sel.loc[sel['r_peak_adarri']==1].index, sel.filtered[sel['r_peak_adarri']==1], zorder=10, c='g')
        plt.scatter(sel.loc[sel['r_peak_physio']==1].index, sel.signal[sel['r_peak_physio']==1], zorder=10, c='b')

        plt.legend(['ECG', 'ECG Filtered', 'Removed by Adarri', 'Biosppy Detection', 'Physionet toolbox Detection'], loc=1)


        plt.tight_layout()



    for idx_removed in physio_missing[10:30]:

        start_idx = idx_removed - 20*fs
        duration = 40
        ecg_peak_detection_plot_compare1()


    # start_idx = 10000
    # duration = 40
    # ecg_peak_detection_plot_compare1()

    # start_idx = 100000
    # duration = 40
    # ecg_peak_detection_plot_compare1()

    # start_idx = 500000
    # duration = 40
    # ecg_peak_detection_plot_compare1()

    # start_idx = 1000000
    # duration = 40
    # ecg_peak_detection_plot_compare1()

    # start_idx = 2000000
    # duration = 40
    # ecg_peak_detection_plot_compare1()
